<a href="https://colab.research.google.com/github/jarvis-2003/Dental_ai/blob/main/Final_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required packages
!pip install -q TotalSegmentator SimpleITK nibabel pydicom
!pip install -q numpy scipy scikit-image trimesh matplotlib plotly pandas xmltodict

print("All dependencies installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.9/544.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 4.9 MB/s eta 0:00:0

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Create directories
paths = {
    'input': '/content/input',
    'output': '/content/output',
    'meshes': '/content/meshes',
}

for path in paths.values():
    Path(path).mkdir(parents=True, exist_ok=True)

print("Directories created!")

for key, path in paths.items():
    Path(path).mkdir(parents=True, exist_ok=True)
    print(f"✓ {key}: {path}")

print("\n✅ Environment configured!")

Tesla T4, 15360 MiB
Directories created!
✓ input: /content/input
✓ output: /content/output
✓ meshes: /content/meshes

✅ Environment configured!


In [7]:
import torch

if torch.cuda.is_available():
    device = "gpu"
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("No GPU detected. Running on CPU.")

print("Selected device:", device)

GPU detected: Tesla T4
Selected device: gpu


In [12]:
import SimpleITK as sitk
import nibabel as nib
from skimage import measure
import trimesh
from typing import Dict, Tuple
import json , os
from totalsegmentator.python_api import totalsegmentator
from totalsegmentator.nifti_ext_header import load_multilabel_nifti


# ============================================================
# 1. DICOM PREPROCESSING
# ============================================================

def convert_dicom_to_nifti(dicom_dir: str, output_nifti_path: str) -> str:
    """
    Convert DICOM series to NIfTI format for nnU-Net.

    Input:  DICOM folder (multiple DICOM files)
    Output: Single 3D NIfTI file (case_ID_0000.nii.gz)
    """
    print(f"📂 Loading DICOM from: {dicom_dir}")

    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(dicom_dir)

    if not series_ids:
        raise ValueError(f"❌ No DICOM series found in {dicom_dir}")

    series_files = reader.GetGDCMSeriesFileNames(dicom_dir, series_ids[0])
    reader.SetFileNames(series_files)

    image = reader.Execute()

    # Print volume info
    print(f"✓ Volume Size: {image.GetSize()} voxels")
    print(f"✓ Voxel Spacing: {image.GetSpacing()} mm")
    print(f"✓ Origin: {image.GetOrigin()}")

    # Save as NIfTI
    sitk.WriteImage(image, output_nifti_path)
    print(f"✓ Saved NIfTI: {output_nifti_path}")

    return output_nifti_path

# ============================================================
# 2. TOTALSEGMENTATOR API INFERENCE
# ============================================================

def run_totalsegmentator(
    input_nifti_path: str,
    output_dir: str,
    task: str
) -> str:

    print(f"\nRunning TotalSegmentator task: {task}")

    os.makedirs(output_dir, exist_ok=True)

    input_img = nib.load(input_nifti_path)

    print("Input shape:", input_img.shape)
    print("Input affine:\n", input_img.affine)

    # Call TotalSegmentator API
    output_img = totalsegmentator(
        input_img,
        task=task,
        ml=True,
        device=device,
        verbose = True
    )

    output_path = os.path.join(
        output_dir,
        f"{task}_segmentation.nii.gz"
    )

    nib.save(output_img, output_path)

    print("Segmentation saved:", output_path)

    # Verify geometry
    seg_img = nib.load(output_path)

    if seg_img.shape != input_img.shape:
        raise ValueError(
            f"Shape mismatch: input={input_img.shape}, "
            f"segmentation={seg_img.shape}"
        )

    if not np.allclose(
        seg_img.affine,
        input_img.affine,
        atol=1e-4
    ):
        raise ValueError("Input and segmentation affines differ!")

    print("Geometry verified.")

    return output_path
# ============================================================
# 3. LABEL EXTRACTION & VISUALIZATION
# ============================================================

def load_segmentation_volume(nifti_path: str) -> Tuple[np.ndarray, Dict]:
    """
    Load segmentation NIfTI file.

    Returns:
    - segmentation_volume: numpy array (Z, Y, X)
    - metadata: dict with spacing, shape, etc.
    """
    print(f"\n📂 Loading segmentation: {nifti_path}")

    nifti_img = nib.load(nifti_path)
    data = nifti_img.get_fdata()
    header = nifti_img.header
    spacing = header.get_zooms()

    print(f"✓ Volume Shape: {data.shape}")
    print(f"✓ Voxel Spacing: {spacing} mm")
    print(f"✓ Data Type: {data.dtype}")
    print(f"✓ Unique Labels: {np.unique(data)}")

    metadata = {
        'shape': data.shape,
        'spacing': spacing,
        'affine': nifti_img.affine,
        'dtype': str(data.dtype),
    }

    return data, metadata


def extract_label_mask(segmentation_volume: np.ndarray, label_value: int) -> np.ndarray:
    """
    Extract binary mask for specific label.
    """
    mask = (segmentation_volume == label_value).astype(np.uint8)
    voxel_count = np.sum(mask)
    print(f"✓ Label {label_value}: {voxel_count} voxels extracted")
    return mask


def visualize_label_slices(segmentation_volume: np.ndarray, label_value: int, title: str):
    """
    Visualize axial, sagittal, coronal slices of a specific label.
    """
    mask = extract_label_mask(segmentation_volume, label_value)

    # Get middle slices
    z_mid = mask.shape[0] // 2
    y_mid = mask.shape[1] // 2
    x_mid = mask.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Label {label_value}: {title}", fontsize=14, fontweight='bold')

    # Axial (Z-axis)
    axes[0].imshow(mask[z_mid, :, :], cmap='hot')
    axes[0].set_title(f"Axial (Z={z_mid})")
    axes[0].axis('off')

    # Sagittal (X-axis)
    axes[1].imshow(mask[:, y_mid, :], cmap='hot')
    axes[1].set_title(f"Sagittal (Y={y_mid})")
    axes[1].axis('off')

    # Coronal (Y-axis)
    axes[2].imshow(mask[:, :, x_mid], cmap='hot')
    axes[2].set_title(f"Coronal (X={x_mid})")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


# ============================================================
# 4. 3D MESH GENERATION
# ============================================================

def convert_mask_to_mesh(
    mask: np.ndarray,
    nifti_affine: np.ndarray,
    label_name: str
) -> trimesh.Trimesh:

    print(f"\nConverting {label_name} to mesh...")

    # Nibabel arrays are normally indexed as (I, J, K).
    # Generate vertices in voxel-index coordinates.
    verts, faces, normals, values = measure.marching_cubes(
        mask,
        level=0.5,
        spacing=(1.0, 1.0, 1.0)
    )

    # Convert voxel indices to NIfTI physical coordinates.
    affine = np.asarray(nifti_affine, dtype=float)

    verts_h = np.column_stack([
        verts,
        np.ones(len(verts))
    ])

    verts_world = (affine @ verts_h.T).T[:, :3]

    # Transform normals correctly using the inverse transpose
    # of the affine's linear component.
    linear = affine[:3, :3]

    normals_world = (
        np.linalg.inv(linear).T @ normals.T
    ).T

    normals_world /= np.maximum(
        np.linalg.norm(normals_world, axis=1, keepdims=True),
        1e-12
    )

    mesh = trimesh.Trimesh(
        vertices=verts_world,
        faces=faces,
        vertex_normals=normals_world,
        process=False
    )

    mesh.merge_vertices()

    try:
        mesh.update_faces(mesh.unique_faces())
        mesh.update_faces(mesh.nondegenerate_faces())
    except AttributeError:
        pass

    mesh.remove_unreferenced_vertices()

    print("Vertices:", len(mesh.vertices))
    print("Faces:", len(mesh.faces))
    print("Bounds:", mesh.bounds)

    return mesh


def export_mesh(mesh: trimesh.Trimesh, output_path: str):
    """
    Export mesh to STL or GLB format.
    """
    ext = Path(output_path).suffix.lower()

    if ext == '.stl':
        mesh.export(output_path)
        print(f"✓ Exported STL: {output_path}")
    elif ext == '.glb':
        mesh.export(output_path)
        print(f"✓ Exported GLB: {output_path}")
    else:
        stl_path = output_path.replace(ext, '.stl')
        mesh.export(stl_path)
        print(f"✓ Exported STL: {stl_path}")

print("✅ All helper functions defined!")

✅ All helper functions defined!


In [5]:
from google.colab import files
import zipfile
import os
import shutil

# ============================================
# Upload DICOM ZIP file
# ============================================

print("📤 Upload your DICOM ZIP file:")

uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded.")

# Get uploaded filename
filename = next(iter(uploaded.keys()))

# Make sure it is a ZIP file
if not filename.lower().endswith(".zip"):
    raise ValueError("❌ Only ZIP files are allowed. Please upload a .zip file.")

print(f"\n📦 Extracting {filename}...")

# Extraction directory
dicom_folder = "/content/dicom_folder"

# Remove old folder if it exists
if os.path.exists(dicom_folder):
    shutil.rmtree(dicom_folder)

os.makedirs(dicom_folder)

# Extract ZIP
try:
    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall(dicom_folder)

except zipfile.BadZipFile:
    raise ValueError("❌ The uploaded file is not a valid ZIP file.")

print("✅ ZIP extracted successfully!")

# ============================================
# List DICOM folder contents
# ============================================

print("\n📂 DICOM folder contents:")

for root, dirs, files_list in os.walk(dicom_folder):
    level = root.replace(dicom_folder, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}📁 {os.path.basename(root)}/")

    for file in files_list[:20]:
        print(f"{indent}    📄 {file}")



📤 Upload your DICOM ZIP file:


Saving dicom_input.zip to dicom_input.zip

📦 Extracting dicom_input.zip...
✅ ZIP extracted successfully!

📂 DICOM folder contents:
📁 dicom_folder/
    📁 dicom_input/
        📄 dicom.dcm


In [9]:
# ============================================
# Convert DICOM → NIfTI
# ============================================
import os
os.makedirs("/content/input", exist_ok=True)

nifti_input = convert_dicom_to_nifti(
    os.path.join(dicom_folder,'dicom_input'),
    "/content/input/case_001_0000.nii.gz"
)

print("\n✅ DICOM → NIfTI conversion completed!")
print(f"📄 NIfTI file: {nifti_input}")

📂 Loading DICOM from: /content/dicom_folder/dicom_input
✓ Volume Size: (568, 568, 334, 1) voxels
✓ Voxel Spacing: (0.3, 0.3, 0.2999999999999991, 1.0) mm
✓ Origin: (0.0, 0.0, 100.2, 0.0)
✓ Saved NIfTI: /content/input/case_001_0000.nii.gz

✅ DICOM → NIfTI conversion completed!
📄 NIfTI file: /content/input/case_001_0000.nii.gz


In [10]:
import nibabel as nib
import numpy as np
import os

img = nib.load(nifti_input)
data = np.asanyarray(img.dataobj)

print("Original shape:", data.shape)

if data.ndim == 4 and data.shape[-1] == 1:
    data = data[..., 0]

if data.ndim != 3:
    raise ValueError(f"Expected 3D CT, got {data.shape}")

img_3d = nib.Nifti1Image(
    data,
    img.affine,
    img.header
)

img_3d.set_qform(img.affine, code=1)
img_3d.set_sform(img.affine, code=1)

nifti_input_3d = "/content/input/case_001_3d.nii.gz"
os.makedirs("/content/input", exist_ok=True)

nib.save(img_3d, nifti_input_3d)

print("Saved 3D CT:", nib.load(nifti_input_3d).shape)

Original shape: (568, 568, 334, 1)
Saved 3D CT: (568, 568, 334)


In [13]:
# ============================================================
# RUN TOTALSEGMENTATOR
# ============================================================

# Run the dental teeth task
teeth_seg_path = run_totalsegmentator(
    input_nifti_path=nifti_input_3d,
    output_dir="/content/output",
    task="teeth"
)

# Run craniofacial structures task
cranio_seg_path = run_totalsegmentator(
    input_nifti_path=nifti_input_3d,
    output_dir="/content/output",
    task="craniofacial_structures"
)

print("Teeth segmentation:", teeth_seg_path)
print("Craniofacial segmentation:", cranio_seg_path)


Running TotalSegmentator task: teeth
Input shape: (568, 568, 334)
Input affine:
 [[ -0.30000001   0.          -0.          -0.        ]
 [  0.          -0.30000001  -0.          -0.        ]
 [  0.           0.          -0.30000001 100.19999695]
 [  0.           0.           0.           1.        ]]
Using Device: cuda:0

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024



Downloading: 100%|██████████| 232M/232M [00:06<00:00, 34.3MB/s]


Download finished. Extracting...
Generating rough segmentation for cropping...
Using Device: cuda:0

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024



Downloading: 100%|██████████| 230M/230M [00:06<00:00, 35.3MB/s]


Download finished. Extracting...
Generating rough segmentation for cropping...


...

sending off prediction to background worker for resampling and export
done with s01
GPU prediction completed. Waiting for remaining segmentation exports to finish...
Segmentation export complete.
  Predicted in 19.99s
Resampling...
  back to original shape: (307, 259, 206)
  Resampled in 0.34s
Undoing canonical...
Undoing cropping...
Segmentation saved: /content/output/teeth_segmentation.nii.gz
Geometry verified.

Running TotalSegmentator task: craniofacial_structures
Input shape: (568, 568, 334)
Input affine:
 [[ -0.30000001   0.          -0.          -0.        ]
 [  0.          -0.30000001  -0.          -0.        ]
 [  0.           0.          -0.30000001 100.19999695]
 [  0.           0.           0.           1.        ]]
Using Device: cuda:0

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
tmp_dir: /tmp/nnunet_tmp_7qxjbcqr
Resampling...
  from shape (568, 568, 334) to shape (28, 28, 17)
  Resampled in

100%|██████████| 1/1 [00:00<00:00, 47.70it/s]


sending off prediction to background worker for resampling and export
done with s01
GPU prediction completed. Waiting for remaining segmentation exports to finish...
Segmentation export complete.
  Predicted in 10.56s
Resampling...
  back to original shape: (568, 568, 334)
  Resampled in 0.51s
Undoing canonical...
Rough organ segmentation generated in 15.27s
tmp_dir: /tmp/nnunet_tmp_c34qq755
  cropping from (568, 568, 334) to (512, 533, 334)
Resampling...
  from shape (512, 533, 334) to shape (307, 320, 200)
  Resampled in 0.76s
Predicting...
There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting s01:
perform_everything_on_device: True


100%|██████████| 36/36 [00:03<00:00,  9.58it/s]


sending off prediction to background worker for resampling and export
done with s01
GPU prediction completed. Waiting for remaining segmentation exports to finish...
Segmentation export complete.
  Predicted in 25.21s
Resampling...
  back to original shape: (512, 533, 334)
  Resampled in 0.98s
Undoing canonical...
Undoing cropping...
Segmentation saved: /content/output/craniofacial_structures_segmentation.nii.gz
Geometry verified.
Teeth segmentation: /content/output/teeth_segmentation.nii.gz
Craniofacial segmentation: /content/output/craniofacial_structures_segmentation.nii.gz


In [14]:
# ============================================================
# LOAD TOTALSEGMENTATOR LABEL MAPS
# ============================================================

def load_totalsegmentator_labels(seg_path):

    seg_img, label_map = load_multilabel_nifti(seg_path)

    seg_data = np.asanyarray(
        seg_img.dataobj
    ).astype(np.int16)

    # Convert label IDs to integer keys
    label_map = {
        int(k): str(v)
        for k, v in label_map.items()
    }

    print("\nSegmentation:", seg_path)
    print("Shape:", seg_data.shape)
    print("Affine:\n", seg_img.affine)

    print("\nAvailable structures:")

    for label_id, label_name in sorted(label_map.items()):
        voxel_count = int(np.count_nonzero(
            seg_data == label_id
        ))

        print(
            f"{label_id}: {label_name} "
            f"({voxel_count:,} voxels)"
        )

    return seg_data, seg_img.affine, label_map


teeth_volume, teeth_affine, teeth_labels = (
    load_totalsegmentator_labels(teeth_seg_path)
)

cranio_volume, cranio_affine, cranio_labels = (
    load_totalsegmentator_labels(cranio_seg_path)
)


Segmentation: /content/output/teeth_segmentation.nii.gz
Shape: (568, 568, 334)
Affine:
 [[ -0.30000001   0.          -0.          -0.        ]
 [  0.          -0.30000001  -0.          -0.        ]
 [  0.           0.          -0.30000001 100.19999695]
 [  0.           0.           0.           1.        ]]

Available structures:
1: lower_jawbone (1,384,546 voxels)
2: upper_jawbone (702,693 voxels)
3: left_inferior_alveolar_canal (12,417 voxels)
4: right_inferior_alveolar_canal (12,846 voxels)
5: left_maxillary_sinus (145,586 voxels)
6: right_maxillary_sinus (141,894 voxels)
7: pharynx (172,483 voxels)
8: bridge (0 voxels)
9: crown (32 voxels)
10: implant (0 voxels)
11: upper_right_central_incisor_fdi11 (18,034 voxels)
12: upper_right_lateral_incisor_fdi12 (11,423 voxels)
13: upper_right_canine_fdi13 (17,892 voxels)
14: upper_right_first_premolar_fdi14 (14,481 voxels)
15: upper_right_second_premolar_fdi15 (12,262 voxels)
16: upper_right_first_molar_fdi16 (0 voxels)
17: upper_right_sec

In [15]:
# ============================================================
# LOAD BOTH TOTALSEGMENTATOR OUTPUTS
# ============================================================

# 1. Load teeth segmentation
teeth_volume, teeth_affine, teeth_labels = (
    load_totalsegmentator_labels(
        "/content/output/teeth_segmentation.nii.gz"
    )
)

# 2. Load craniofacial segmentation
cranio_volume, cranio_affine, cranio_labels = (
    load_totalsegmentator_labels(
        "/content/output/craniofacial_structures_segmentation.nii.gz"
    )
)

# ============================================================
# SUMMARY
# ============================================================

print("\n========== TEETH SEGMENTATION ==========")
print("Shape:", teeth_volume.shape)

for label_id, label_name in sorted(teeth_labels.items()):
    count = np.count_nonzero(teeth_volume == label_id)
    print(f"{label_id}: {label_name} ({count:,} voxels)")


print("\n========== CRANIOFACIAL SEGMENTATION ==========")
print("Shape:", cranio_volume.shape)

for label_id, label_name in sorted(cranio_labels.items()):
    count = np.count_nonzero(cranio_volume == label_id)
    print(f"{label_id}: {label_name} ({count:,} voxels)")


Segmentation: /content/output/teeth_segmentation.nii.gz
Shape: (568, 568, 334)
Affine:
 [[ -0.30000001   0.          -0.          -0.        ]
 [  0.          -0.30000001  -0.          -0.        ]
 [  0.           0.          -0.30000001 100.19999695]
 [  0.           0.           0.           1.        ]]

Available structures:
1: lower_jawbone (1,384,546 voxels)
2: upper_jawbone (702,693 voxels)
3: left_inferior_alveolar_canal (12,417 voxels)
4: right_inferior_alveolar_canal (12,846 voxels)
5: left_maxillary_sinus (145,586 voxels)
6: right_maxillary_sinus (141,894 voxels)
7: pharynx (172,483 voxels)
8: bridge (0 voxels)
9: crown (32 voxels)
10: implant (0 voxels)
11: upper_right_central_incisor_fdi11 (18,034 voxels)
12: upper_right_lateral_incisor_fdi12 (11,423 voxels)
13: upper_right_canine_fdi13 (17,892 voxels)
14: upper_right_first_premolar_fdi14 (14,481 voxels)
15: upper_right_second_premolar_fdi15 (12,262 voxels)
16: upper_right_first_molar_fdi16 (0 voxels)
17: upper_right_sec

In [16]:
# ============================================================
# GENERATE MESHES FROM TOTALSEGMENTATOR
# ============================================================

meshes = {}

def generate_all_meshes(
    segmentation_volume,
    nifti_affine,
    label_map,
    prefix
):

    for label_id, label_name in label_map.items():

        if label_id == 0:
            continue

        print(f"\nProcessing: {label_name}")

        mask = (
            segmentation_volume == label_id
        ).astype(np.uint8)

        voxel_count = int(np.count_nonzero(mask))

        if voxel_count < 100:
            print("Skipping small structure")
            continue

        try:
            mesh = convert_mask_to_mesh(
                mask,
                nifti_affine,
                label_name
            )

            mesh_key = f"{prefix}_{label_id}"

            meshes[mesh_key] = {
                "mesh": mesh,
                "name": label_name,
                "label_id": label_id,
                "source": prefix,
                "vertices": len(mesh.vertices),
                "faces": len(mesh.faces)
            }

        except Exception as e:
            print(
                f"Failed to create {label_name}: {e}"
            )


generate_all_meshes(
    segmentation_volume=teeth_volume,
    nifti_affine=teeth_affine,
    label_map=teeth_labels,
    prefix="teeth"
)

# Process craniofacial segmentation
generate_all_meshes(
    segmentation_volume=cranio_volume,
    nifti_affine=cranio_affine,
    label_map=cranio_labels,
    prefix="cranio"
)

print(f"\nGenerated {len(meshes)} meshes")


Processing: lower_jawbone

Converting lower_jawbone to mesh...
Vertices: 257727
Faces: 515830
Bounds: [[-141.75000563 -100.650004     21.44999382]
 [ -49.65000197  -34.35000136   83.24999627]]

Processing: upper_jawbone

Converting upper_jawbone to mesh...
Vertices: 221129
Faces: 442516
Bounds: [[-130.35000518  -92.25000367   57.74999526]
 [ -55.05000219  -35.55000141   83.24999627]]

Processing: left_inferior_alveolar_canal

Converting left_inferior_alveolar_canal to mesh...
Vertices: 9324
Faces: 18644
Bounds: [[-141.75000563  -94.05000374   22.34999385]
 [-112.35000446  -47.55000189   60.44999537]]

Processing: right_inferior_alveolar_canal

Converting right_inferior_alveolar_canal to mesh...
Vertices: 9022
Faces: 18040
Bounds: [[ -70.65000281 -100.650004     25.04999396]
 [ -53.55000213  -52.65000209   57.74999526]]

Processing: left_maxillary_sinus

Converting left_maxillary_sinus to mesh...
Vertices: 31059
Faces: 62118
Bounds: [[-128.2500051   -81.75000325   67.34999564]
 [-100.3

In [17]:
# ============================================================
# EXPORT ALL MESHES (STL + GLB)
# ============================================================

print("=" * 70)
print("EXPORTING MESHES")
print("=" * 70)

from pathlib import Path

output_mesh_dir = Path("/content/meshes")
output_mesh_dir.mkdir(parents=True, exist_ok=True)

for mesh_key, mesh_data in meshes.items():

    mesh = mesh_data["mesh"]
    label_name = mesh_data["name"]

    # Include task prefix to avoid duplicate filenames
    prefix = mesh_data.get("source", "segmentation")

    safe_name = (
        label_name
        .replace(" & ", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .lower()
    )

    filename = f"{prefix}_{safe_name}"

    # Export STL
    stl_path = output_mesh_dir / f"{filename}.stl"
    export_mesh(mesh, str(stl_path))

    # Export GLB
    glb_path = output_mesh_dir / f"{filename}.glb"
    mesh.export(str(glb_path))

    print(f"✓ Exported GLB: {glb_path}")

print("\nAll meshes exported!")

print("\nGenerated files:")
!ls -lh /content/meshes/

EXPORTING MESHES
✓ Exported STL: /content/meshes/teeth_lower_jawbone.stl
✓ Exported GLB: /content/meshes/teeth_lower_jawbone.glb
✓ Exported STL: /content/meshes/teeth_upper_jawbone.stl
✓ Exported GLB: /content/meshes/teeth_upper_jawbone.glb
✓ Exported STL: /content/meshes/teeth_left_inferior_alveolar_canal.stl
✓ Exported GLB: /content/meshes/teeth_left_inferior_alveolar_canal.glb
✓ Exported STL: /content/meshes/teeth_right_inferior_alveolar_canal.stl
✓ Exported GLB: /content/meshes/teeth_right_inferior_alveolar_canal.glb
✓ Exported STL: /content/meshes/teeth_left_maxillary_sinus.stl
✓ Exported GLB: /content/meshes/teeth_left_maxillary_sinus.glb
✓ Exported STL: /content/meshes/teeth_right_maxillary_sinus.stl
✓ Exported GLB: /content/meshes/teeth_right_maxillary_sinus.glb
✓ Exported STL: /content/meshes/teeth_pharynx.stl
✓ Exported GLB: /content/meshes/teeth_pharynx.glb
✓ Exported STL: /content/meshes/teeth_upper_right_central_incisor_fdi11.stl
✓ Exported GLB: /content/meshes/teeth_upper

In [18]:
import pandas as pd

print("=" * 70)
print("MESH STATISTICS")
print("=" * 70)

stats = []

for label, mesh_data in meshes.items():
    mesh = mesh_data['mesh']
    name = mesh_data['name']

    # Calculate metrics
    surface_area = mesh.area
    volume = mesh.volume if mesh.is_watertight else "N/A"
    bbox = mesh.bounds
    bbox_volume = np.prod(bbox[1] - bbox[0])

    stats.append({
        'Label': label,
        'Structure': name,
        'Vertices': mesh_data['vertices'],
        'Faces': mesh_data['faces'],
        'Surface Area (mm²)': f"{surface_area:.1f}",
        'Volume (mm³)': f"{volume:.1f}" if isinstance(volume, (int, float)) else volume,
        'Watertight': mesh.is_watertight,
    })

# Create DataFrame
df = pd.DataFrame(stats)
print("\n" + df.to_string(index=False))

# Export as CSV
csv_path = '/content/meshes/mesh_statistics.csv'
df.to_csv(csv_path, index=False)
print(f"\n✓ Statistics saved: {csv_path}")

MESH STATISTICS

   Label                               Structure  Vertices   Faces Surface Area (mm²) Volume (mm³)  Watertight
 teeth_1                           lower_jawbone    257727  515830            19323.6      37382.2        True
 teeth_2                           upper_jawbone    221129  442516            16312.6          N/A       False
 teeth_3            left_inferior_alveolar_canal      9324   18644              675.8        331.0        True
 teeth_4           right_inferior_alveolar_canal      9022   18040              656.2        343.2        True
 teeth_5                    left_maxillary_sinus     31059   62118             2302.8       3927.7        True
 teeth_6                   right_maxillary_sinus     32204   64524             2383.1       3830.4        True
 teeth_7                                 pharynx     32535   65074             2562.4       4653.4        True
teeth_11       upper_right_central_incisor_fdi11      6886   13764              511.0        48

In [20]:
import nibabel as nib
import numpy as np
from scipy import ndimage
from pathlib import Path

segmentation_files = {
    "teeth": "/content/output/teeth_segmentation.nii.gz",
    "craniofacial": (
        "/content/output/"
        "craniofacial_structures_segmentation.nii.gz"
    )
}

for task_name, seg_path in segmentation_files.items():

    print("\n" + "=" * 70)
    print(f"ANALYZING: {task_name.upper()}")
    print("=" * 70)

    if not Path(seg_path).exists():
        print("Missing file:", seg_path)
        continue

    seg_img = nib.load(seg_path)
    seg = np.asanyarray(seg_img.dataobj)

    if seg.ndim == 4 and seg.shape[-1] == 1:
        seg = seg[..., 0]

    if seg.ndim != 3:
        print("Unexpected segmentation shape:", seg.shape)
        continue

    # Voxel volume from the NIfTI affine (mm³)
    voxel_volume_mm3 = abs(
        np.linalg.det(seg_img.affine[:3, :3])
    )

    print("Shape:", seg.shape)
    print("Voxel volume:", voxel_volume_mm3, "mm³")

    # Label statistics
    print("\nLABEL VOLUME ANALYSIS")

    for label in np.unique(seg):
        label = int(label)

        # Skip background
        if label == 0:
            continue

        mask = seg == label
        voxel_count = int(np.count_nonzero(mask))
        volume_mm3 = voxel_count * voxel_volume_mm3

        print(
            f"Label {label}: "
            f"{voxel_count:,} voxels, "
            f"{volume_mm3:,.1f} mm³"
        )

        # Bounding box in voxel coordinates
        coords = np.where(mask)

        print(
            "  Voxel bounds:",
            [(int(c.min()), int(c.max())) for c in coords]
        )

        # Connected components
        _, num_components = ndimage.label(mask)

        print("  Connected components:", num_components)

print("\nAnalysis complete!")


ANALYZING: TEETH
Shape: (568, 568, 334)
Voxel volume: 0.027000003218650946 mm³

LABEL VOLUME ANALYSIS
Label 1: 1,384,546 voxels, 37,382.7 mm³
  Voxel bounds: [(166, 472), (115, 335), (57, 262)]
  Connected components: 14
Label 2: 702,693 voxels, 18,972.7 mm³
  Voxel bounds: [(184, 434), (119, 307), (57, 141)]
  Connected components: 41
Label 3: 12,417 voxels, 335.3 mm³
  Voxel bounds: [(375, 472), (159, 313), (133, 259)]
  Connected components: 1
Label 4: 12,846 voxels, 346.8 mm³
  Voxel bounds: [(179, 235), (176, 335), (142, 250)]
  Connected components: 1
Label 5: 145,586 voxels, 3,930.8 mm³
  Voxel bounds: [(335, 427), (162, 272), (57, 109)]
  Connected components: 3
Label 6: 141,894 voxels, 3,831.1 mm³
  Voxel bounds: [(189, 275), (184, 297), (57, 104)]
  Connected components: 7
Label 7: 172,483 voxels, 4,657.0 mm³
  Voxel bounds: [(246, 392), (187, 335), (57, 174)]
  Connected components: 4
Label 9: 32 voxels, 0.9 mm³
  Voxel bounds: [(283, 288), (119, 124), (158, 161)]
  Connect

In [21]:
from pathlib import Path
import numpy as np
import nibabel as nib

print("\n" + "=" * 70)
print("COLAB DENTAL SEGMENTATION - COMPLETE SUMMARY")
print("=" * 70)

# --------------------------------------------------
# 1. INPUT DATA
# --------------------------------------------------

input_file = "/content/input/case_001_0000.nii.gz"

input_img = nib.load(input_file)
input_shape = input_img.shape

# Physical voxel spacing from affine
voxel_spacing = nib.affines.voxel_sizes(input_img.affine)

# --------------------------------------------------
# 2. SEGMENTATION OUTPUTS
# --------------------------------------------------

teeth_seg_path = (
    "/content/output/teeth_segmentation.nii.gz"
)

cranio_seg_path = (
    "/content/output/"
    "craniofacial_structures_segmentation.nii.gz"
)

segmentation_files = {
    "Teeth": teeth_seg_path,
    "Craniofacial structures": cranio_seg_path,
}

segmentation_stats = {}

for task_name, seg_path in segmentation_files.items():

    if Path(seg_path).exists():

        seg_img = nib.load(seg_path)
        seg_data = np.asanyarray(seg_img.dataobj)

        if seg_data.ndim == 4 and seg_data.shape[-1] == 1:
            seg_data = seg_data[..., 0]

        unique_labels = np.unique(seg_data)
        foreground_labels = unique_labels[unique_labels != 0]

        segmentation_stats[task_name] = {
            "labels": len(foreground_labels),
            "shape": seg_data.shape,
        }

    else:
        segmentation_stats[task_name] = None

# --------------------------------------------------
# 3. MESH STATISTICS
# --------------------------------------------------

mesh_count = len(meshes) if "meshes" in globals() else 0

mesh_dir = Path("/content/meshes")

stl_files = list(mesh_dir.glob("*.stl"))
glb_files = list(mesh_dir.glob("*.glb"))

# --------------------------------------------------
# 4. DEVICE INFORMATION
# --------------------------------------------------

device_used = globals().get("device", "Not recorded")

# --------------------------------------------------
# 5. BUILD REPORT
# --------------------------------------------------

summary_report = f"""
COLAB DENTAL SEGMENTATION - COMPLETE SUMMARY

==================================================
INPUT DATA
==================================================

Format: NIfTI (.nii.gz)
Input path: {input_file}
Volume size: {input_shape}
Voxel spacing (mm): {voxel_spacing}

==================================================
MODEL INFERENCE
==================================================

Model: TotalSegmentator
Framework: TotalSegmentator / nnU-Net-based inference
Device selected: {device_used}

Tasks:
- teeth
- craniofacial_structures

==================================================
SEGMENTATION OUTPUTS
==================================================
"""

for task_name, seg_path in segmentation_files.items():

    stats = segmentation_stats[task_name]

    summary_report += f"\n{task_name}:\n"
    summary_report += f"  Path: {seg_path}\n"

    if stats is not None:
        summary_report += (
            f"  Volume shape: {stats['shape']}\n"
            f"  Foreground labels: {stats['labels']}\n"
        )
    else:
        summary_report += "  Status: File not found\n"

summary_report += f"""

==================================================
MESH EXPORTS
==================================================

Meshes generated in memory: {mesh_count}
STL files exported: {len(stl_files)}
GLB files exported: {len(glb_files)}

Mesh directory: {mesh_dir}

==================================================
PROCESSING STEPS
==================================================

- DICOM to NIfTI conversion
- TotalSegmentator inference
- Multilabel segmentation processing
- Label-based mask extraction
- Marching Cubes mesh generation
- Mesh optimization and export

==================================================
NEXT STEPS
==================================================

1. Verify segmentation masks in 3D Slicer.
2. Verify mesh alignment against the original CBCT.
3. Import STL/GLB files into the Three.js viewer.
4. Integrate segmentation results into the Django backend.
5. Validate anatomical measurements before clinical use.

==================================================
"""

print(summary_report)

# --------------------------------------------------
# 6. SAVE REPORT
# --------------------------------------------------

mesh_dir.mkdir(parents=True, exist_ok=True)

report_path = mesh_dir / "REPORT.txt"

with open(report_path, "w") as f:
    f.write(summary_report)

print(f"Report saved to: {report_path}")


COLAB DENTAL SEGMENTATION - COMPLETE SUMMARY

COLAB DENTAL SEGMENTATION - COMPLETE SUMMARY

INPUT DATA

Format: NIfTI (.nii.gz)
Input path: /content/input/case_001_0000.nii.gz
Volume size: (568, 568, 334, 1)
Voxel spacing (mm): [0.30000001 0.30000001 0.30000001]

MODEL INFERENCE

Model: TotalSegmentator
Framework: TotalSegmentator / nnU-Net-based inference
Device selected: gpu

Tasks:
- teeth
- craniofacial_structures

SEGMENTATION OUTPUTS

Teeth:
  Path: /content/output/teeth_segmentation.nii.gz
  Volume shape: (568, 568, 334)
  Foreground labels: 58

Craniofacial structures:
  Path: /content/output/craniofacial_structures_segmentation.nii.gz
  Volume shape: (568, 568, 334)
  Foreground labels: 6


MESH EXPORTS

Meshes generated in memory: 63
STL files exported: 63
GLB files exported: 63

Mesh directory: /content/meshes

PROCESSING STEPS

- DICOM to NIfTI conversion
- TotalSegmentator inference
- Multilabel segmentation processing
- Label-based mask extraction
- Marching Cubes mesh g

In [22]:
from google.colab import files
import shutil
from pathlib import Path

print("Preparing dental segmentation results...")

# Create a package directory
package_dir = Path("/content/dental_results")
package_dir.mkdir(parents=True, exist_ok=True)

# 1. Copy STL and GLB meshes
mesh_dir = Path("/content/meshes")

if mesh_dir.exists():
    shutil.copytree(
        mesh_dir,
        package_dir / "meshes",
        dirs_exist_ok=True
    )

# 2. Copy both segmentation NIfTI files
segmentation_files = {
    "teeth_segmentation.nii.gz":
        "/content/output/teeth_segmentation.nii.gz",

    "craniofacial_structures_segmentation.nii.gz":
        "/content/output/craniofacial_structures_segmentation.nii.gz",
}

seg_dir = package_dir / "segmentations"
seg_dir.mkdir(exist_ok=True)

for filename, source_path in segmentation_files.items():

    source = Path(source_path)

    if source.exists():
        shutil.copy2(source, seg_dir / filename)
        print(f"Included segmentation: {filename}")
    else:
        print(f"WARNING: Missing segmentation: {source_path}")

# 3. Create ZIP
zip_path = shutil.make_archive(
    "/content/dental_results",
    "zip",
    root_dir=package_dir
)

print("\nZIP created:", zip_path)

# 4. Download
files.download(zip_path)

Preparing dental segmentation results...
Included segmentation: teeth_segmentation.nii.gz
Included segmentation: craniofacial_structures_segmentation.nii.gz

ZIP created: /content/dental_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>